# From Prototype to Production: Deploying Your First AI Agent

Welcome, future AI architect! In this notebook, you'll embark on a journey from building a simple, local AI agent to understanding how to make it a real, usable application.

We'll focus on two key concepts, simplified for beginners:

1.  **Agent Interoperability**: How to make specialized agents work together as a team.
2.  **Deployment Architecture**: How to expose your agent to the world so others can use it, just like a real web service.

We'll be using the `google-adk` (Agent Development Kit) to build our agents. Let's get started!


## Theory Primer: Multi-Agent Teams

### From Solo Artist to Newsroom

For the last four days, you built single agents — one brain, one loop, one set of tools. That works beautifully for small jobs. But ask a single agent to *research a market, verify the facts, and write a polished 1,000-word report*, and you'll watch it wobble: it forgets step one by step three, mixes research notes into final prose, and produces something vaguely mediocre.

Think of a newsroom. You don't hire one person to be reporter, fact-checker, editor, and publisher. You hire specialists and give them a workflow. That's exactly what a **multi-agent team** is: several focused agents, each with a narrow job, passing work down the line.

### Why Specialisation Wins

Every agent has a **context window** — its short-term memory. A generalist agent burns that memory juggling instructions for four different jobs at once. A specialist agent spends all of it doing *one* job brilliantly.

Three things define each teammate:

- **Role** — the job title ("Senior Research Analyst")
- **Goal** — the single outcome they own ("Find 5 credible, recent sources on the topic")
- **Backstory** — the personality and expertise that shapes their tone and judgement

That trio is the agent's job description. Vague job descriptions produce vague employees — and the same is true here.

### Meet Today's Team

| Agent | Analogy | Owns |
|---|---|---|
| **Manager** | The Editor-in-Chief | Breaks the request into tasks, delegates, checks quality |
| **Researcher** | The Reporter | Uses search tools to gather raw facts and sources |
| **Writer** | The Copywriter | Turns messy notes into a clean, structured report |

The Manager never Googles. The Researcher never writes prose. The Writer never invents facts. **Clean boundaries create clean output.**

### How Work Actually Flows

Two orchestration patterns matter today:

1. **Sequential** — an assembly line. Researcher finishes → hands notes to Writer → done. Predictable, cheap, easy to debug. Start here.
2. **Hierarchical** — a manager agent decides *who* does *what*, and can send work back for revision. More powerful, more expensive, more unpredictable.

The magic ingredient is the **handoff**: the output of one agent becomes the input (context) of the next. If the Researcher returns sloppy notes, the Writer inherits that sloppiness. Garbage in, garbage out — just with extra steps.

### Watch Out For

- **Runaway loops** — agents endlessly delegating to each other. Always set a max iteration limit.
- **Token cost** — three agents means roughly three times the API spend. Keep prompts tight.
- **Overlapping roles** — if two agents can do the same job, they'll duplicate work or contradict each other.

### Where This Is Heading

Your capstone **Universal Knowledge Worker** is just this newsroom, scaled up: a team that can research *anything*, reason about it, and deliver a usable artefact. Today you build the skeleton. You've got this — let's hire your first team. 🚀


In [ ]:
!pip install -q google-adk wikipedia wikipedia

import os
from google.colab import userdata

print("✅ Dependencies installed.")

# --- Configure Your API Key ---
# To use the Gemini model, you need a Google AI API key.
# 1. Get your key from Google AI Studio: https://aistudio.google.com/app/api-keys
# 2. In this Colab notebook, click the 'Key' icon on the left sidebar.
# 3. Click 'Add new secret'.
# 4. Name the secret 'GOOGLE_API_KEY' and paste your key in the 'Value' field.

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
    print("✅ GOOGLE_API_KEY configured successfully!")
except userdata.SecretNotFoundError:
    print("🛑 Secret 'GOOGLE_API_KEY' not found. Please add it to Colab Secrets to proceed.")
except Exception as e:
    print(f"An error occurred: {e}")


## Step 1: Making Agents Talk - Agent Interoperability

Imagine you're building a company. You wouldn't hire one person to do marketing, finance, and engineering, right? You'd hire specialists and have them work together.

AI agents are the same! A single, monolithic agent that tries to do everything often performs poorly. The modern approach is to build a **multi-agent system** where small, specialized agents collaborate.

**The Architecture:**
*   **Delegator Agent**: A "manager" agent that understands the main goal.
*   **Specialist Agent**: A "team member" with a specific skill or tool (e.g., a researcher, a calculator, a database expert).

When the Delegator Agent gets a request it can't handle alone, it delegates the task to the appropriate Specialist Agent. The `google-adk` makes this easy by allowing one agent to be used as a `sub_agent` (like a tool) by another.


In [ ]:
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini

# --- 1. The Specialist: Researcher Agent ---
# This agent has a specific tool: searching internal documents.

def search_internal_docs(query: str) -> str:
    """Searches the internal company documentation for a specific project name."""
    print(f"\n🔎 [ResearcherAgent is searching for: '{query}']")
    internal_db = {
        "Project Phoenix": "Project Phoenix is a next-gen AI initiative focused on multi-agent systems.",
        "Project Apollo": "Project Apollo is our new cloud infrastructure deployment plan."
    }
    return internal_db.get(query, f"Sorry, I could not find any documents for '{query}'.")

researcher_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    name="ResearcherAgent",
    description="An agent that can search internal company documents.",
    tools=[search_internal_docs]
)

print("✅ Specialist 'ResearcherAgent' created.")

# --- 2. The Manager: Delegator Agent ---
# This agent's job is to delegate tasks to its sub-agents.

delegator_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    name="DelegatorAgent",
    description="A manager agent that answers questions by delegating to specialist agents.",
    instruction="""
    You are a helpful manager. Your goal is to answer questions about internal company projects.
    To do this, you MUST use the 'ResearcherAgent' to find the information.
    Do not try to make up answers.
    """,
    sub_agents=[researcher_agent] # <-- This is how we build the team!
)

print("✅ Manager 'DelegatorAgent' created.")

# --- 3. Run the System ---
# We ask the manager a question. It will automatically delegate to the researcher.
print("\n--- Asking the manager a question... ---")
user_question = "What can you tell me about Project Phoenix?"
print(f"👤 User: {user_question}")

# The .run() method executes the agent's logic
response = delegator_agent.run(user_question)

print(f"\n🤖 Manager Response:\n{response.content}")


## Step 2: Exposing Your Agent - A Taste of Deployment

An agent that only lives in a notebook is a prototype. To make it a real product, you need to **deploy** it.

**Deployment** means packaging your agent's code and running it on a server, making it accessible over the internet through an **API** (Application Programming Interface). This turns your agent into a service that other applications (like a website, a mobile app, or a chatbot) can call.

**The Architecture:**

`Client Application` ---> `Internet (API Request)` ---> `Your Deployed Agent`

Setting up servers and APIs can be complex. To understand the core concept, we will **simulate** a deployment. We'll wrap our agent in a simple Python function that acts like an API endpoint. It will:
1.  Receive a request (as a Python dictionary).
2.  Run the agent.
3.  Return a response (as a Python dictionary).


In [ ]:
# --- 1. The Agent to be Deployed ---
# A simple agent that acts as a creative writer.
creative_writer_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="You are a creative writer. Write a single, compelling tagline for the given product."
)

print("✅ 'CreativeWriterAgent' created.")

# --- 2. The Simulated API Endpoint ---
# This function pretends to be a web server endpoint.
def creative_writer_api(request: dict) -> dict:
    """
    This function simulates a REST API endpoint.
    It takes a request dictionary and returns a response dictionary.
    """
    print(f"\n▶️ API endpoint received request: {request}")
    product_name = request.get("product")
    
    if not product_name:
        return {"error": "'product' not found in request."}

    # The agent runs to process the request
    response = creative_writer_agent.run(f"Product: {product_name}")

    # The API formats the agent's output into a structured response
    api_response = {"tagline": response.content}
    print(f"◀️ API endpoint sending response: {api_response}")
    return api_response

print("✅ Mock API endpoint 'creative_writer_api' defined.")

# --- 3. Simulate a Client Calling the API ---
print("\n--- A client is calling our API... ---")
client_request = {"product": "A smart coffee mug that keeps your drink at the perfect temperature"}

api_result = creative_writer_api(client_request)

print("\n--- Client received the following data from the API ---")
print(api_result)


## The Grand Finale: Your Personal Universal Knowledge Worker

Now, let's combine everything we've learned to build a final, powerful agent: a **Personal Universal Knowledge Worker**.

This agent will:
1.  **Have a powerful tool**: It will use Google Search to find real-time information on the web.
2.  **Have a clear mission**: Its instructions will guide it to act as a helpful researcher.
3.  **Be production-ready**: We will wrap it in our simulated API endpoint architecture, showing how it could be deployed as a real service.


In [ ]:
import wikipedia

def search_wikipedia(topic: str) -> dict:
    """Searches Wikipedia for real-world facts and summaries."""
    print(f"\n🔍 [Agent is searching Wikipedia for: {topic}]")
    try:
        return {"status": "success", "result": wikipedia.summary(topic, sentences=3)}
    except Exception:
        return {"status": "error", "error_message": f"Could not find information on {topic}."}

research_assistant_agent = LlmAgent(
    model=Gemini(model="gemini-1.5-flash-latest"),
    instruction="""
    You are a world-class research assistant. Your goal is to answer the user's question accurately and concisely.
    Use the search_wikipedia tool to find the most relevant, up-to-date information.
    Synthesize the information from the search results into a clear answer.
    """,
    tools=[search_wikipedia]
)
print("✅ Personal Research Assistant agent created with Wikipedia Tool.")

# --- 3. Create the Final API Endpoint ---
def research_assistant_api(request: dict) -> dict:
    if not research_assistant_agent:
        return {"error": "Agent not initialized. Check previous cell for errors."}
    
    print(f"\n▶️ Research API received query: '{request.get('query')}'")
    query = request.get("query")
    if not query:
        return {"error": "'query' not found in request."}

    response = research_assistant_agent.run(query)
    
    api_response = {"answer": response.content}
    print(f"◀️ Research API sending response.")
    return api_response

# --- 4. Simulate a Final Client Call ---
if research_assistant_agent:
    print("\n--- Calling the final Research Assistant API... ---")
    final_client_request = {"query": "What are the main benefits of using a multi-agent system in AI development?"}
    final_answer = research_assistant_api(final_client_request)

    print("\n--- Final Answer from Research Assistant --- ")
    print(final_answer.get('answer'))
